# FileID import for read/write

Start `examples/python/fileTransfer/testDevice.py` in another process, then run this notebook. It imports a caller-owned file into the target device persistence manager and passes the returned `FileID` to channels that accept file values.

In [ ]:
import tempfile
import uuid

import stipy


config = stipy.Configuration({
    "Device Name": "FileID Import Notebook Client",
    "IP Address": "localhost",
    "Module": "0",
    "Target Server": "sr-magis/2/Frame2",
})

config.set("NetworkHub", "NameService", "192.168.88.252:2809")
config.set("omniORB", "traceLevel", "0")
config.set("omniORB", "scanGranularity", "1")
config.set("omniORB", "clientConnectTimeOutPeriod", "500")
config.set("omniORB", "clientCallTimeOutPeriod", "2000")

device_id = stipy.DeviceID("FileTransferDevice", "localhost", 0, "sr-magis/2/Frame2")
device = stipy.connect(device_id, config=config)
persistence = device.getPersistenceManager()
target_file_server = persistence.getFileServer()

assert persistence is not None
assert target_file_server is not None

device

Create a caller-side source file and serve it from a virtual file server. The target import call receives both the `FileID` and the `FileServer` that can provide the bytes.

In [ ]:
payload = (
    b"fileTransfer FileID import example\n"
    b"The target device should receive this through PersistenceManager.importFile().\n"
)

source_server = persistence.makeVirtualFileServer()
source_holder = persistence.makeFileHolder(
    tempfile.gettempdir(),
    f"fileTransfer-fileid-import-source-{uuid.uuid4().hex}.txt",
)

assert source_server is not None
assert source_holder is not None
assert source_holder.openFile()
try:
    assert source_holder.writeBytes(payload)
finally:
    source_holder.closeFile()

source_id = source_holder.getID()
assert source_server.addFile(source_holder)
assert source_server.findFile(source_id)
assert source_server.getFileSize(source_id) == len(payload)

source_id.filename, source_server.getFileSize(source_id)

Importing copies the file immediately into the target persistence manager. The imported `FileID` is the value to pass to `device.write()` or parameterized `device.read()`.

In [ ]:
options = stipy.ImportFileOptions()
assert options.storage == stipy.ImportStorage.DiskTemporary

imported = persistence.importFile(source_id, source_server, options)
assert imported is not None

with imported:
    imported_id = imported.fileID
    assert target_file_server.findFile(imported_id)
    assert target_file_server.getFileSize(imported_id) == len(payload)

    assert device.write(6, imported_id)
    assert device.read(20, imported_id) == len(payload)

assert imported.closed
assert not target_file_server.findFile(imported_id)

imported_id.filename, len(payload)